In [1]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
import sklearn as skt
import xgboost as xgb
import json
from scipy.stats import trim_mean
import math
from pandas.api.types import ( is_numeric_dtype, is_categorical_dtype, is_object_dtype, is_datetime64_any_dtype )
from default_risk.scripts.cv_mlfow_integration import run_cv_tracked_mlflow
import default_risk.config as cfg
import os
import xgboost as xgb
from dotenv import load_dotenv
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
import mlflow
import mlflow.xgboost
import dtale


previous_application_df = pd.read_parquet(cfg.CLEANS_DIR / "previous_application.train-cleaned.parquet")



In [3]:
previous_application_df.head(5)

,id_prev,id_curr,name_contract_type,amt_annuity,amt_annuity_and_cnt_payment_are_missing,amt_application,amt_credit,amt_down_payment,amt_down_payment_is_missing,amt_goods_price,...,days_first_due_has_sentinel_value,days_first_due,days_last_due_1st_version_has_sentinel_value,days_last_due_1st_version,days_last_due_has_sentinel_value,days_last_due,days_termination_has_sentinel_value,days_termination,nflag_insured_on_approval,days_and_insurance_information_are_missing
0,2030495,271877,Consumer loans,1730.430,0,17145.0,17145.0,0.0,0,17145.0,...,0,-42.0,0,300.0,0,-42.0,0,-37.0,0.0,0
1,2802425,108129,Cash loans,25188.615,0,607500.0,679671.0,NaN,1,607500.0,...,0,-134.0,0,916.0,1,NaN,1,NaN,1.0,0
2,2523466,122040,Cash loans,15060.735,0,112500.0,136444.5,NaN,1,112500.0,...,0,-271.0,0,59.0,1,NaN,1,NaN,1.0,0
3,2819243,176158,Cash loans,47041.335,0,450000.0,470790.0,NaN,1,450000.0,...,0,-482.0,0,-152.0,0,-182.0,0,-177.0,1.0,0
4,1784265,202054,Cash loans,31924.395,0,337500.0,404055.0,NaN,1,337500.0,...,0,NaN,0,NaN,0,NaN,0,NaN,NaN,1


In [4]:
#creating columns before aggregation
previous_application_df["diff_application_credit"] = previous_application_df["amt_application"] - previous_application_df["amt_credit"]
previous_application_df["ratio_credit_to_goods"] = previous_application_df["amt_credit"] / (previous_application_df["amt_goods_price"].replace(0,np.nan))
previous_application_df["total_interest_charged"] = (previous_application_df["amt_annuity"] * previous_application_df["cnt_payment"]) - previous_application_df["amt_credit"]
previous_application_df["ratio_credit_to_annuity"]= previous_application_df["amt_credit"] / (previous_application_df["amt_annuity"].replace(0,np.nan))


In [5]:
instalament_df = pd.read_parquet(cfg.PROCESSED_DIR / "installments_payments.train-processed.parquet")
previous_application_df= previous_application_df.merge(instalament_df,how="left",on= "id_prev")
previous_application_df.head()

,id_prev,id_curr,name_contract_type,amt_annuity,amt_annuity_and_cnt_payment_are_missing,amt_application,amt_credit,amt_down_payment,amt_down_payment_is_missing,amt_goods_price,...,instalments_repeated_for_payment_in_advance_sum,instalments_is_delinquency_mean,instalments_is_delinquency_sum,instalments_days_of_delinquency_mean,instalments_days_of_delinquency_max,instalments_days_of_delinquency_sum,instalments_days_in_advance_mean,instalments_days_in_advance_max,instalments_days_in_advance_sum,instalments_completion_ratio
0,2030495,271877,Consumer loans,1730.430,0,17145.0,17145.0,0.0,0,17145.0,...,0.0,0.000000,0.0,0.000000,0.0,0.0,0.000000,0.0,0.0,1.0
1,2802425,108129,Cash loans,25188.615,0,607500.0,679671.0,NaN,1,607500.0,...,0.0,0.000000,0.0,0.000000,0.0,0.0,9.200000,11.0,46.0,1.0
2,2523466,122040,Cash loans,15060.735,0,112500.0,136444.5,NaN,1,112500.0,...,0.0,0.111111,1.0,0.111111,1.0,1.0,8.333333,12.0,75.0,1.0
3,2819243,176158,Cash loans,47041.335,0,450000.0,470790.0,NaN,1,450000.0,...,0.0,0.000000,0.0,0.000000,0.0,0.0,7.090909,30.0,78.0,1.0
4,1784265,202054,Cash loans,31924.395,0,337500.0,404055.0,NaN,1,337500.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [6]:

previous_application_to_pivot_df= previous_application_df.drop(columns="id_prev")
mask_no_final= previous_application_df["flag_last_application_for_the_contract"] == "N"
previous_application_to_pivot_df= previous_application_to_pivot_df.loc[~mask_no_final]
previous_application_to_pivot_df= previous_application_to_pivot_df.drop(columns=["flag_last_application_for_the_contract"])

previous_application_to_pivot_df.sort_values(["id_curr","days_decision"],inplace=True,ascending=False)
last_three= previous_application_to_pivot_df.groupby("id_curr").head(3)

In [7]:
last_three = last_three.copy()
last_three["loan_order"] = last_three.groupby("id_curr").cumcount() + 1
df_wide = last_three.pivot(index="id_curr", columns="loan_order")
df_wide.columns =[f"{col}_prev_{rank}" for col, rank in df_wide.columns]
df_wide= df_wide.reset_index()
dtale.show(df_wide)

2026-06-11 00:28:10,474 - ERROR    - Exception on /health [GET]
Traceback (most recent call last):
  File "c:\Users\kuroc\OneDrive\Escritorio\default risk\default-risk\Lib\site-packages\flask\app.py", line 2529, in wsgi_app
    response = self.full_dispatch_request()
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\kuroc\OneDrive\Escritorio\default risk\default-risk\Lib\site-packages\flask\app.py", line 1825, in full_dispatch_request
    rv = self.handle_user_exception(e)
         ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\kuroc\OneDrive\Escritorio\default risk\default-risk\Lib\site-packages\flask\app.py", line 1821, in full_dispatch_request
    rv = self.preprocess_request()
         ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\kuroc\OneDrive\Escritorio\default risk\default-risk\Lib\site-packages\flask\app.py", line 2313, in preprocess_request
    rv = self.ensure_sync(before_func)()
         ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\kuroc\OneDrive\Escritorio\defa

In [8]:
previous_application_df["name_contract_status"]= previous_application_df["name_contract_status"].str.lower()
previous_application_df["code_reject_reason"]= previous_application_df["code_reject_reason"].str.lower()
previous_application_df= pd.get_dummies(previous_application_df, columns=["name_contract_status","code_reject_reason"])
dtale.show(previous_application_df.head(5))

In [9]:



#we will calculate aggregations per client for differents tables so we will separate dictionaries per table
agg_from_prev_app_dict= {

    #saving the ammount of contract
    "id_prev" : ["count"],
    
    #for log transformated we want to catch the mean and the std (avoiding the impact of the heavy tail from this columns)
    "log_amt_credit": ["mean","std"],   
    "log_amt_application": ["mean","std"],
    "log_amt_down_payment": ["mean","std"],
    "log_amt_goods_price": ["mean","std"],
    "log_amt_annuity": ["mean","std"],
    "log_total_interest_charged" : ["mean","std"],

    #for non transformated columns we want to catch the representative values and the acumulated
    "amt_credit": ["max", "min","median","sum"],
    "amt_application": ["max", "min","median","sum"],
    "amt_down_payment": ["max", "min","median","sum"],
    "amt_goods_price": ["max", "min","median","sum"],
    "amt_annuity": ["max", "min","median"],
    "total_interest_charged": ["max", "min","median"],

    #others_monetary
    "diff_application_credit": ["max","mean","min","median","sum"],
    "log_diff_application_credit": ["max","mean","min"],
    "rate_down_payment": ["max","mean","std","min","median"],
    "ratio_credit_to_goods" : ["max","mean","std","min","median"],
    "ratio_credit_to_annuity" : ["max","mean","std","min","median"],

    #categoricals
    "code_reject_reason_limit" : ["sum"],
    "name_contract_status_approved": ["mean","sum"],
    "name_contract_status_canceled": ["mean","sum"],
    "name_contract_status_refused": ["mean","sum"],
    "amt_annuity_and_cnt_payment_are_missing" :["mean","sum"],
    "amt_down_payment_is_missing" : ["mean","sum"],
    "nflag_insured_on_approval" : ["mean","sum"],
    "days_and_insurance_information_are_missing": ["mean","sum"],
    "amt_goods_price_is_missing" : ["mean","sum"],
    "rate_down_payment_is_missing" : ["mean","sum"],


    #counters
    "days_decision":["mean","min","max"],
    "cnt_payment":["mean","min","max","sum"]
}

In [10]:
agg_from_instalment_payment_dict= {
    "instalments_potentially_on_going" : ["sum"],
    "instalments_is_potentially_incomplete_sequence" : ["mean","sum"],
    "instalments_dead_tail_length" : ["mean","max"],
    "instalments_amt_instalment_sum" : ["mean","sum","max"],
    "instalments_amt_payment_sum" : ["mean", "sum", "max"],
    "instalments_days_of_delinquency_max": ["max"],
    "instalments_days_of_delinquency_mean": ["mean", "max"],
    "instalments_days_in_advance_max":["max"],
    "instalments_days_in_advance_mean": ["mean", "max"],
    "instalments_is_delinquency_sum" : ["sum"] ,
    "instalments_is_delinquency_mean" : ["mean"],
    "instalments_repeated_for_underpayment_sum" : ["sum"],
    "instalments_repeated_for_underpayment_mean" : ["mean"],
    "instalments_repeated_for_reschedule_sum" : ["sum"],
    "instalments_repeated_for_reschedule_mean" : ["mean"],
    "instalments_diff_expected_received_sum" : ["sum"]
}


In [11]:
final_dict_for_agg= agg_from_prev_app_dict | agg_from_instalment_payment_dict

In [12]:

previous_application_df["log_amt_credit"] = np.log1p(previous_application_df["amt_credit"])
previous_application_df["log_amt_application"] = np.log1p(previous_application_df["amt_application"])
previous_application_df["log_amt_annuity"] = np.log1p(previous_application_df["amt_annuity"])
previous_application_df["log_amt_down_payment"] = np.log1p(previous_application_df["amt_down_payment"])
previous_application_df["log_amt_goods_price"] = np.log1p(previous_application_df["amt_goods_price"])
previous_application_df["log_diff_application_credit"] = previous_application_df["log_amt_application"] - previous_application_df["log_amt_credit"]
previous_application_df["log_total_interest_charged"] = np.log1p(previous_application_df["total_interest_charged"])



agg_metrics_df= previous_application_df.groupby("id_curr").agg(final_dict_for_agg)




c:\Users\kuroc\OneDrive\Escritorio\default risk\default-risk\Lib\site-packages\pandas\core\arraylike.py:399: RuntimeWarning: invalid value encountered in log1p
  result = getattr(ufunc, method)(*inputs, **kwargs)


In [13]:
agg_metrics_df.columns= [f"{col[0]}_{col[1]}" for col in agg_metrics_df.columns]
agg_metrics_df= agg_metrics_df.reset_index()

In [14]:
agg_metrics_df.rename(columns={"id_prev_count": "applications_count"})
agg_metrics_df.head()


,id_curr,id_prev_count,log_amt_credit_mean,log_amt_credit_std,log_amt_application_mean,log_amt_application_std,log_amt_down_payment_mean,log_amt_down_payment_std,log_amt_goods_price_mean,log_amt_goods_price_std,...,instalments_days_in_advance_max_max,instalments_days_in_advance_mean_mean,instalments_days_in_advance_mean_max,instalments_is_delinquency_sum_sum,instalments_is_delinquency_mean_mean,instalments_repeated_for_underpayment_sum_sum,instalments_repeated_for_underpayment_mean_mean,instalments_repeated_for_reschedule_sum_sum,instalments_repeated_for_reschedule_mean_mean,instalments_diff_expected_received_sum_sum
0,100002,1,12.095454,NaN,12.095454,NaN,0.000000,NaN,12.095454,NaN,...,31.0,20.421053,20.421053,0.0,0.000000,0.0,0.000000,0.0,0.0,0.000
1,100003,3,12.580207,1.370403,12.526196,1.297500,4.418623,6.248876,12.526196,1.297500,...,14.0,7.448413,11.166667,0.0,0.000000,0.0,0.000000,0.0,0.0,0.000
2,100004,1,9.908823,NaN,10.097532,NaN,8.488999,NaN,10.097532,NaN,...,11.0,7.666667,7.666667,0.0,0.000000,0.0,0.000000,0.0,0.0,0.000
3,100006,9,8.369355,6.360517,8.368877,6.349295,9.505589,2.272189,12.553316,1.211353,...,77.0,25.300000,48.400000,0.0,0.000000,0.0,0.000000,0.0,0.0,0.000
4,100007,6,11.563999,1.274969,11.525923,1.165810,8.125540,0.119429,11.525923,1.165810,...,31.0,4.064803,8.705882,16.0,0.287912,3.0,0.043956,0.0,0.0,29857.365


In [15]:
agg_metrics_df.rename(columns={"id_prev_count": "applications_count"},inplace=True)
previous_application_ready_to_merge= df_wide.merge(agg_metrics_df,on="id_curr",how="left")

In [16]:
cols_to_fix = [
    "instalments_potentially_on_going_sum",
    "instalments_is_potentially_incomplete_sequence_sum",
    "instalments_is_potentially_incomplete_sequence_mean",
    "instalments_is_delincuency_sum",
    "instalments_is_delincuency_mean"
]

for col in cols_to_fix:
    if col in previous_application_ready_to_merge.columns:
        previous_application_ready_to_merge[col] = pd.to_numeric(previous_application_ready_to_merge[col], errors='coerce')

In [18]:

previous_application_ready_to_merge.to_parquet(cfg.PROCESSED_DIR / "prev_app_agg_installments.parquet", index=False)

In [19]:
instalament_df = pd.read_parquet(cfg.PROCESSED_DIR / "installments_payments.train-processed-2.parquet")
instalament_df.head()

,id_prev,instalments_id_curr,instalments_raw_size_serie,instalments_dead_tail_length,instalments_potentially_on_going,instalments_amount_of_versions_in_sequence,instalments_mean_ratio_last_year,instalments_mean_ratio_last_two_years,instalments_mean_historical_payment_rate,instalments_payment_tendence,...,instalments_is_delinquency_mean,instalments_is_delinquency_sum,instalments_days_of_delinquency_mean,instalments_days_of_delinquency_max,instalments_days_of_delinquency_sum,instalments_days_in_advance_mean,instalments_days_in_advance_max,instalments_days_in_advance_sum,instalments_days_of_underpayment_max,instalments_completion_ratio
0,1000001,158271,2,0,False,2,1.0,1.000000,1.000000,0.000000,...,0.000000,0,0.000000,0.0,0.0,16.000000,26.0,32.0,NaN,1.000000
1,1000003,252457,3,0,False,1,1.0,1.000000,1.000000,0.000000,...,0.000000,0,0.000000,0.0,0.0,15.333333,17.0,46.0,NaN,1.000000
2,1000004,260094,7,0,False,2,1.0,1.153314,1.051894,-0.153314,...,0.000000,0,0.000000,0.0,0.0,26.714286,58.0,187.0,NaN,1.000000
3,1000005,176456,11,0,False,1,NaN,NaN,0.909091,NaN,...,0.181818,2,0.363636,3.0,4.0,8.818182,36.0,97.0,-1448.0,0.909027
4,1000007,256657,5,0,True,1,1.0,1.000000,1.000000,0.000000,...,0.000000,0,0.000000,0.0,0.0,16.800000,21.0,84.0,NaN,1.000000


In [2]:
previous_application_df = pd.read_parquet(cfg.CLEANS_DIR / "previous_application.train-cleaned.parquet")
#creating columns before aggregation
previous_application_df["diff_application_credit"] = previous_application_df["amt_application"] - previous_application_df["amt_credit"]
previous_application_df["ratio_credit_to_goods"] = previous_application_df["amt_credit"] / (previous_application_df["amt_goods_price"].replace(0,np.nan))
previous_application_df["total_interest_charged"] = (previous_application_df["amt_annuity"] * previous_application_df["cnt_payment"]) - previous_application_df["amt_credit"]
previous_application_df["implied_interest_rate"] = (previous_application_df["amt_annuity"] * previous_application_df["cnt_payment"]) / previous_application_df["amt_credit"].replace(0, np.nan)
previous_application_df["ratio_credit_to_annuity"]= previous_application_df["amt_credit"] / (previous_application_df["amt_annuity"].replace(0,np.nan))
instalament_df = pd.read_parquet(cfg.PROCESSED_DIR / "installments_payments.train-processed-2.parquet")



#results = instalament_df[["instalments_id_curr","instalments_mean_ratio_last_year", "instalments_mean_ratio_last_two_years","instalments_mean_historical_payment_rate","instalments_payment_tendence"]].drop_duplicates()

#instalament_df=instalament_df.drop(columns=["instalments_mean_ratio_last_year","instalments_mean_ratio_last_two_years","instalments_mean_historical_payment_rate","instalments_payment_tendence","instalments_id_curr"])



previous_application_df= previous_application_df.merge(instalament_df,how="left",on= "id_prev")
previous_application_to_pivot_df= previous_application_df.drop(columns="id_prev")
mask_no_final= previous_application_df["flag_last_application_for_the_contract"] == "N"
previous_application_to_pivot_df= previous_application_to_pivot_df.loc[~mask_no_final]
previous_application_to_pivot_df= previous_application_to_pivot_df.drop(columns=["flag_last_application_for_the_contract"])

previous_application_to_pivot_df.sort_values(["id_curr","days_decision"],inplace=True,ascending=False)
last_three= previous_application_to_pivot_df.groupby("id_curr").head(3)
last_three = last_three.copy()
last_three["loan_order"] = last_three.groupby("id_curr").cumcount() + 1
df_wide = last_three.pivot(index="id_curr", columns="loan_order")
df_wide.columns =[f"{col}_prev_{rank}" for col, rank in df_wide.columns]
df_wide= df_wide.reset_index()
#previous_application_df["name_contract_status"]= previous_application_df["name_contract_status"].str.lower()
#previous_application_df["code_reject_reason"]= previous_application_df["code_reject_reason"].str.lower()
#previous_application_df["name_contract_type"] = previous_application_df["name_contract_type"].str.lower()
previous_application_df = pd.get_dummies(previous_application_df, columns=[ "code_reject_reason"])

#we will calculate aggregations per client for differents tables so we will separate dictionaries per table
agg_from_prev_app_dict= {

    #saving the ammount of contract
    "id_prev" : ["count"],
    
    #for log transformated we want to catch the mean and the std (avoiding the impact of the heavy tail from this columns)
    "log_amt_credit": ["mean","std"],   
    "log_amt_application": ["mean","std"],
    "log_amt_down_payment": ["mean","std"],
    "log_amt_goods_price": ["mean","std"],
    "log_amt_annuity": ["mean","std"],
    "log_total_interest_charged" : ["mean","std"],
    "instalments_completion_ratio" : ["mean","std"],
 
    

    #for non transformated columns we want to catch the representative values and the acumulated
    "amt_credit": ["max", "min","median","sum"],
    "amt_application": ["max", "min","median","sum"],
    "amt_down_payment": ["max", "min","median","sum"],
    "amt_goods_price": ["max", "min","median","sum"],
    "amt_annuity": ["max", "min","median"],
    "total_interest_charged": ["max", "min","median"],
    "implied_interest_rate" : ["max", "min","mean"],
    

    #others_monetary
    "diff_application_credit": ["max","mean","min","median","sum"],
    "log_diff_application_credit": ["max","mean","min"],
    "rate_down_payment": ["max","mean","std","min","median"],
    "ratio_credit_to_goods" : ["max","mean","std","min","median"],
    "ratio_credit_to_annuity" : ["max","mean","std","min","median"],

    #categoricals
    #"code_reject_reason_limit" : ["sum"],
    #"code_reject_reason_hc" : ["sum"],
    #"code_reject_reason_sco" : ["sum"],
    #"code_reject_reason_client" : ["sum"],
    #"name_contract_status_approved": ["mean","sum"],
    #"name_contract_status_canceled": ["mean","sum"],
    #"name_contract_status_refused": ["mean","sum"],
    #"name_contract_type_cash loans": ["mean", "sum"], 
    #"name_contract_type_consumer loans": ["mean", "sum"],
    #"name_contract_type_revolving loans": ["mean", "sum"], 
    "amt_annuity_and_cnt_payment_are_missing" :["mean","sum"],
    "amt_down_payment_is_missing" : ["mean","sum"],
    "nflag_insured_on_approval" : ["mean","sum"],
    "days_and_insurance_information_are_missing": ["mean","sum"],
    "amt_goods_price_is_missing" : ["mean","sum"],
    "rate_down_payment_is_missing" : ["mean","sum"],


    #counters
    "days_decision":["mean","min","max"],
    "cnt_payment":["mean","min","max","sum"]
}

agg_from_instalment_payment_dict= {
    "instalments_amount_of_versions_in_sequence" : ["mean","max","sum"],
    "instalments_potentially_on_going" : ["sum"],
    "instalments_dead_tail_length" : ["mean","max"],
    "instalments_amt_instalment_sum" : ["mean","sum","max"],
    "instalments_amt_payment_sum" : ["mean", "sum", "max"],
    "instalments_days_of_delinquency_max": ["max"],
    "instalments_days_of_delinquency_mean": ["mean", "max"],
    "instalments_extra_instalament_sum":["sum"],
    "instalments_extra_instalament_mean":["mean"],
    "instalments_days_in_advance_max":["max"],
    "instalments_days_of_underpayment_max":["max"],
    "instalments_days_in_advance_mean": ["mean", "max"],
    "instalments_is_delinquency_sum" : ["sum"] ,
    "instalments_is_delinquency_mean" : ["mean"],
    "instalments_diff_expected_received_sum" : ["sum"]
}

final_dict_for_agg= agg_from_prev_app_dict | agg_from_instalment_payment_dict


previous_application_df["log_amt_credit"] = np.log1p(previous_application_df["amt_credit"])
previous_application_df["log_amt_application"] = np.log1p(previous_application_df["amt_application"])
previous_application_df["log_amt_annuity"] = np.log1p(previous_application_df["amt_annuity"])
previous_application_df["log_amt_down_payment"] = np.log1p(previous_application_df["amt_down_payment"])
previous_application_df["log_amt_goods_price"] = np.log1p(previous_application_df["amt_goods_price"])
previous_application_df["log_diff_application_credit"] = previous_application_df["log_amt_application"] - previous_application_df["log_amt_credit"]
previous_application_df["log_total_interest_charged"] = np.log1p(previous_application_df["total_interest_charged"])



agg_metrics_df= previous_application_df.groupby("id_curr").agg(final_dict_for_agg)

agg_metrics_df.columns= [f"{col[0]}_{col[1]}" for col in agg_metrics_df.columns]

agg_metrics_df.rename(columns={"id_prev_count": "applications_count"},inplace=True)
agg_metrics_df= agg_metrics_df.reset_index()

agg_metrics_df["global_approval_ratio"] = agg_metrics_df["amt_credit_sum"] / agg_metrics_df["amt_application_sum"].replace(0, np.nan)
agg_metrics_df["days_decision_spread"] = agg_metrics_df["days_decision_max"] - agg_metrics_df["days_decision_min"]

previous_application_ready_to_merge= df_wide.merge(agg_metrics_df,on="id_curr",how="left")

cols_to_fix = [
    "instalments_potentially_on_going_sum",
    "instalments_is_potentially_incomplete_sequence_sum",
    "instalments_is_potentially_incomplete_sequence_mean",
    "instalments_is_delinquency_sum",
    "instalments_is_delinquency_mean"
]



for col in cols_to_fix:
    if col in previous_application_ready_to_merge.columns:
        previous_application_ready_to_merge[col] = pd.to_numeric(previous_application_ready_to_merge[col], errors='coerce')

#previous_application_ready_to_merge= previous_application_ready_to_merge.merge(results, left_on="id_curr",right_on="instalments_id_curr",how="left")

previous_application_ready_to_merge.to_parquet(cfg.PROCESSED_DIR / "prev_app_agg_installments-2.parquet", index=False)

c:\Users\kuroc\OneDrive\Escritorio\default risk\default-risk\Lib\site-packages\pandas\core\arraylike.py:399: RuntimeWarning: invalid value encountered in log1p
  result = getattr(ufunc, method)(*inputs, **kwargs)


In [24]:
previous_app_final = pd.read_parquet(cfg.PROCESSED_DIR / "prev_app_agg_installments-2.parquet")
dtale.show(previous_app_final.head(5))

2026-06-15 22:58:32,573 - ERROR    - Exception on /health [GET]
Traceback (most recent call last):
  File "c:\Users\kuroc\OneDrive\Escritorio\default risk\default-risk\Lib\site-packages\flask\app.py", line 2529, in wsgi_app
    response = self.full_dispatch_request()
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\kuroc\OneDrive\Escritorio\default risk\default-risk\Lib\site-packages\flask\app.py", line 1825, in full_dispatch_request
    rv = self.handle_user_exception(e)
         ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\kuroc\OneDrive\Escritorio\default risk\default-risk\Lib\site-packages\flask\app.py", line 1821, in full_dispatch_request
    rv = self.preprocess_request()
         ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\kuroc\OneDrive\Escritorio\default risk\default-risk\Lib\site-packages\flask\app.py", line 2313, in preprocess_request
    rv = self.ensure_sync(before_func)()
         ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\kuroc\OneDrive\Escritorio\defa

In [9]:
previous_application_df = pd.read_parquet(cfg.CLEANS_DIR / "previous_application.train-cleaned.parquet")
instalament_df = pd.read_parquet(cfg.PROCESSED_DIR / "installments_payments.train-processed-2.parquet")
what= previous_application_df.merge(instalament_df,how="left",on= "id_prev")
dtale.show(what.head(50))

2026-06-13 22:37:48,872 - ERROR    - Exception on /health [GET]
Traceback (most recent call last):
  File "c:\Users\kuroc\OneDrive\Escritorio\default risk\default-risk\Lib\site-packages\flask\app.py", line 2529, in wsgi_app
    response = self.full_dispatch_request()
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\kuroc\OneDrive\Escritorio\default risk\default-risk\Lib\site-packages\flask\app.py", line 1825, in full_dispatch_request
    rv = self.handle_user_exception(e)
         ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\kuroc\OneDrive\Escritorio\default risk\default-risk\Lib\site-packages\flask\app.py", line 1821, in full_dispatch_request
    rv = self.preprocess_request()
         ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\kuroc\OneDrive\Escritorio\default risk\default-risk\Lib\site-packages\flask\app.py", line 2313, in preprocess_request
    rv = self.ensure_sync(before_func)()
         ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\kuroc\OneDrive\Escritorio\defa